**偶发信号**和和**多探针覆盖范围下的智能设备信号**是需要清洗的内容，
- 偶发信号的特征为数据量极少且到达探针的数量少
- 多探针覆盖范围下的智能设备信号特征为数据量多且到达探针的数量为1-4个


另外主要还有**常驻人群**，**游客**，**通勤人群**三类数据，但由于信号干扰等原因，不同数据之间的分隔基准并不明确，同时人群种类可能并不仅仅只有前文所列的三类，不同类型也有可能因为信号干扰、行为模式不同而导致数据差异，因此通过简单的逻辑判断清洗效果和分类效果都不甚理想。

尝试通过kmeans或dbscan聚类来进行分类

## DBSCAN聚类
DBSCAN（Density-Based Spatial Clustering of Applications with Noise）是一种基于密度的聚类算法，可以将数据点分成不同的簇，并且能够识别噪声点（不属于任何簇的点）。

DBSCAN聚类算法的基本思想是：

在给定的数据集中，根据每个数据点周围其他数据点的密度情况，将数据点分为核心点、边界点和噪声点。

- 核心点是周围某个半径内有足够多其他数据点的数据点；
- 边界点是不满足核心点要求，但在某个核心点的半径内的数据点；
- 噪声点则是不满足任何条件的点。


## 轮廓系数、卡林斯基-哈拉巴斯指数、DBCV进行评估

- **轮廓系数（silhouette_score）**，轮廓系数指标不关注样本的实际类别，而是通过分析聚类结果中样本的内聚度和分离度两种因素来给出成绩，取值范围为（-1，1），**值越大代表聚类的结果越合理**。
- **卡林斯基-哈拉巴斯指数(Calinski-Harabasz Index)**，组内离散程度低，**协方差矩阵**的迹就会越小，同时，组间离散程度大，**协方差矩阵**的迹就会越大。因此calinski_harabasz**指数越高越好**。
  
 $$ s(k)=\frac{T_r(B_k)}{T_r(W_k)}*\frac{N-k}{k-1} $$

其中$N$是数据集的样本量，$k$为簇的个数，$B_k$是组间离散矩阵,即不同簇之间的协方差矩阵。$W_k$是簇内离散矩阵，即一个簇内数据的协方差矩阵，而$Tr$表示矩阵的迹。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import matplotlib.cm as cm
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio

import sys
sys.path.append('../plot/')
import myplot

sys.path.append('../utils/')
import utils
import DBSCAN
from DBSCAN import My_DBSCAN,My_DBSCAN_MATRIX

# 设置plotly默认主题
pio.templates.default = 'plotly_white'

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

from collections import namedtuple

from importlib import reload

In [2]:
df_cluster = pd.read_csv(os.getcwd()+'/dacang/cleaned_data/dacang_cleaned_data_advanced.csv')
df_cluster

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-20,178,229,137,47,142",17.02,4.0,38.0,"20,178,229,137,47,142",7.5,20.33
2,"7.6-4,207,140,11,128,186",23.98,4.0,947.0,"4,207,140,11,128,186",7.6,12.00
3,"7.6-208,151,254,40,222,169",23.97,4.0,910.0,"208,151,254,40,222,169",7.6,11.98
4,"7.5-48,74,38,133,29,213",17.00,5.0,49.0,"48,74,38,133,29,213",7.5,20.35
...,...,...,...,...,...,...,...
1063,"8.7-176,70,146,156,89,69",9.38,5.0,179.0,"176,70,146,156,89,69",8.7,17.18
1064,"8.7-44,220,173,148,169,213",7.48,4.0,171.0,"44,220,173,148,169,213",8.7,17.82
1065,"8.7-124,161,119,201,50,40",9.78,6.0,260.0,"124,161,119,201,50,40",8.7,18.98
1066,"8.7-152,207,83,118,160,172",19.45,4.0,54.0,"152,207,83,118,160,172",8.7,12.45


In [ ]:
#多次聚类，确定效果最佳的eps与min_samples
results = My_DBSCAN_MATRIX(df_cluster,
                 init_eps = 0.13,
                 step_eps = 0.01,
                 epoch_eps = 1,
                 init_mSamples = 3,
                 step_mSamples = 1,
                 epoch_mSamples = 1
                 )

In [ ]:
#保存聚类数据
df_dbscan_result = pd.DataFrame({
    "cluster_num":list(map(lambda x : x.cluster_num, results)),
    "noise_num":list(map(lambda x : x.noise_num, results)),
    "dbcv":list(map(lambda x : x.dbcv, results)),
    "silhouette":list(map(lambda x : x.silhouette, results)),
    "calinski":list(map(lambda x : x.calinski, results)),
    "eps":list(map(lambda x : x.eps, results)),
    "min_samples":list(map(lambda x : x.min_samples, results)),
    })

df_dbscan_result.to_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1130.csv",index=False)
#np.save(os.getcwd()+"/dacang/cluster/cluster_mat_result_1128.npy",cluster_labels)

### 第一次聚类结果评估

In [ ]:
def Cut_Data_By_NoiseNum(df,cut_thre):
    df_result = df.copy()
    for index,row in df_result.iterrows():
        if row.noise_num > cut_thre :
            df_result.at[index,'dbcv'] = -1
            df_result.at[index,'cluster_count'] = 0
            df_result.at[index,'silhouette'] = -1
            df_result.at[index,'calinski_harabasz'] = 0
            df_result.at[index,'noise_num'] = 0
    return df_result

def add_data(df,col_name,list):
    min_samples_unique = df.min_samples.unique()
    df_3d = pd.DataFrame(columns=min_samples_unique)
    df_3d.insert(0,'eps',[])
    eps = df.eps.unique()
    min_samples = df.min_samples
    for i in range(len(eps)):
        df_now = (df[df.eps == eps[i]])
        add_dict = {"eps":eps[i]}
        for index,row in df_now.iterrows():
            add_dict.update({row.min_samples:row[col_name]})
        
        df_3d = df_3d._append(add_dict,ignore_index=True)

    Sur_Data = namedtuple("Sur_Data",['values','xAxes','yAxes','name'])
    data = Sur_Data(
                    values=df_3d.drop('eps',axis=1).values,
                    xAxes=df_3d.columns[1:],
                    yAxes=df_3d.eps,
                    name=col_name)
    list.append(data)
    
    # myplot.Surface3D(df_3d.drop('eps',axis=1).values,df_3d.eps,df_3d.columns[1:],
    #                 x_name = 'eps',y_name = "min_samples")

def ShowResult(df,col_name_list):
    data_list = []
    for name in col_name_list:
        add_data(df,name,data_list)
    myplot.Surface3D_supPlot(data_list)

In [ ]:
#读取聚类数据
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise.csv")
cluster_labels =  np.load(os.getcwd()+"/dacang/cluster/cluster_mat_result.npy")
df_result = Cut_Data_By_NoiseNum(df_dbscan_result,800)

In [ ]:
ShowResult(df_dbscan_result,['dbcv','noise_num'])

### 第二次聚类数据评估

In [ ]:
#读取聚类数据
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1128.csv")
cluster_labels =  np.load(os.getcwd()+"/dacang/cluster/cluster_mat_result_1128.npy")

In [ ]:
df_result = Cut_Data_By_NoiseNum(df_dbscan_result,800)

In [ ]:
ShowResult(df_dbscan_result,['dbcv','noise_num','cluster_count','silhouette','calinski_harabasz'])

### 第三次聚类结果评估

In [ ]:
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1129.csv")
ShowResult(df_dbscan_result,['dbcv','noise_num','cluster_count'])

## 针对第一次评估的断崖进行分类测试
eps = 0.74

In [ ]:
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.Double_Axes_Line(df_dbscan_now,"eps","silhouette","calinski_harabasz",
                        xAxes_name = "领域半径(eps)",
                        line1_name = "轮廓系数",
                        line2_name = "卡林斯基-哈拉巴斯指数")

轮廓系数与卡林斯基-哈拉巴斯指数都是内部聚类评价指标，即在不知道真实labels时对聚类结果进行评价。

内部聚类评价指标的一个通病是默认数据集中的类别是簇状结构。

事实上，真实世界的数据集很多都不是簇状结构，所以内部聚类评价往往并不客观。

对于非簇状结构的聚类结果的评价往往使用外部聚类评价指标，常见的有NMI，AMI，F-score，Accuracy等。

In [ ]:
#聚类数量折线图
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.One_Axes_Line(df_dbscan_now,"eps","cluster_count",
                     xAxes_name = "领域半径(eps)",
                     line_name = "聚类数量")

In [ ]:
#噪声数量
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.One_Axes_Line(df_dbscan_now,"eps","noise_num",
                     xAxes_name = "领域半径(eps)",
                     line_name = "噪声数量")

## 确定eps和min_samples后分析

In [ ]:
#eps = 16 , min_samples = 0.73


## test
-----

In [ ]:
from sklearn.cluster import DBSCAN
#if 'label' is in df_cluster.col
df_cluster = df_cluster.drop('label',axis = 1)
dbscan = DBSCAN(eps=0.62, min_samples=10)
cluster = dbscan.fit(df_cluster)
len(set(cluster.labels_))
df_cluster['label'] = cluster.labels_

In [ ]:
from collections import Counter
Counter(cluster.labels_)

Counter({0: 213,
         1: 589,
         2: 411,
         -1: 94,
         3: 243,
         4: 18,
         5: 89,
         6: 148,
         7: 27,
         8: 46,
         9: 15,
         10: 8,
         12: 7,
         11: 12,
         13: 9,
         14: 7,
         15: 8,
         16: 6})

In [ ]:
myplot.Date_Mac_3D_scatter(df_cluster,'Dur','Count','A_count','label')